# RALU

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as dsets
import torchvision.models as tv_models
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as GridSpec
from matplotlib.colors import LinearSegmentedColormap
import numpy as np
from sklearn.metrics import (roc_curve, auc, precision_recall_curve,
                             average_precision_score, confusion_matrix,
                             classification_report, f1_score)
from sklearn.preprocessing import label_binarize
import seaborn as sns
import os, shutil, time, json, warnings, gc
import pandas as pd
from PIL import Image
from collections import defaultdict
from math import pi
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score

warnings.filterwarnings('ignore')
from google.colab import drive
import shutil  # for Colab file manipulation

#####################################################################################
# Mount your Drive first to enable you save the training results
from google.colab import drive
drive.mount('/content/drive')
#####################################################################################

# ============================================================================
#  CONFIGURATION
# ============================================================================
DATASET_TYPE   = 'tinyimagenet200'        # 'cifar10' | 'cifar100' | 'tinyimagenet200' | 'custom'
MODEL_TYPE     = 'resnet34' # 'shallowcnn' | 'vgg16' | 'densenet121' | 'mobilenetv3' | 'vitb16' | 'efficientnetb0' | 'resnet34'
OPTIMIZER_TYPE = 'sgd'  # Options: 'sgd', 'adam', 'adamw', 'adagrad', 'adamax', 'nadam', 'rmsprop', 'rprop', 'adadelta'

# --- Activations to compare ---
ACTIVATION_FUNCTIONS = ['ralu', 'prelu', 'swish', 'relu', 'leaky_relu', 'gelu', 'mish']#'raglu'
#ACTIVATION_FUNCTIONS = ['ralu']#, 'prelu', 'swish', 'relu', 'leaky_relu', 'gelu', 'mish']

# Training hyperparameters
BATCH_SIZE               = 128
NUM_EPOCHS               = 50
LEARNING_RATE            = 0.01
EARLY_STOPPING           = True
EARLY_STOPPING_PATIENCE  = 5

# Mixed-precision training (AMP) — speeds up training on compatible GPUs
USE_AMP = True  # set False if you hit precision issues

# Custom dataset paths (used only when DATASET_TYPE == 'custom')
CUSTOM_DATA_PATH = r'E:\datasets\Cashew'
TRAIN_DIR        = os.path.join(CUSTOM_DATA_PATH, 'train_set')
VALID_DIR        = os.path.join(CUSTOM_DATA_PATH, 'valid_set')
TEST_DIR         = os.path.join(CUSTOM_DATA_PATH, 'test_set')

# TinyImageNet
TINYIMAGENET_DIR = '/content/tiny-imagenet-200'

# Output
BASE_OUTPUT_DIR = '/content/drive/MyDrive/ACTIVATION_FXTNS_RESULTS'

# ============================================================================
#  CROSS-VALIDATION CONFIGURATION
# ============================================================================
RUN_CV_COMPARISON       = True
NUMBER_OF_FOLDS         = 5 #5
CV_EPOCHS               = 40 #40
CV_LEARNING_RATE        = 0.01

CV_ACTIVATION_FUNCTIONS = ['ralu', 'prelu', 'swish', 'relu', 'leaky_relu', 'gelu', 'mish']#'raglu'
#CV_ACTIVATION_FUNCTIONS = ['ralu', 'prelu']#, 'swish', 'relu', 'leaky_relu', 'gelu', 'mish']

print(f"\n{'='*60}")
print(f"Model: {MODEL_TYPE.upper()}")
print(f"DATASET: {DATASET_TYPE.upper()}")
print(f"SINGLE MODEL NUM EPOCHS: {NUM_EPOCHS}")
print(f"CV NUM EPOCHS: {CV_EPOCHS}")
print(f"BATCH SIZE: {BATCH_SIZE}")
print(f"SINGLE MODEL LEARNING RATE: {LEARNING_RATE}")
print(f"CV LEARNING RATE: {CV_LEARNING_RATE}")
print(f"OPTIMIZER: {OPTIMIZER_TYPE.upper()}")
print(f"EARLY STOPPING PATIENCE: {EARLY_STOPPING_PATIENCE}")
print(f"AMP (Mixed Precision): {USE_AMP}")
print(f"{'='*60}\n")
# ============================================================================
#  RALU
# ============================================================================
class RALU(nn.Module):
    """f(x) = x + β · x² / (1 + |x|)"""
    def __init__(self, beta=0.9, learnable=True):
        super().__init__()
        if learnable:
            self.beta = nn.Parameter(torch.tensor(float(beta)))
        else:
            self.register_buffer('beta', torch.tensor(float(beta)))

    def forward(self, x):
        return x + self.beta * (x * x) / (1 + x.abs())


##############################################################################
#  ACTIVATION FACTORY
# ============================================================================
def get_activation(name: str) -> nn.Module:
    name = name.lower()
    mapping = {
        'relu':       nn.ReLU(),
        'prelu':      nn.PReLU(),
        'leaky_relu': nn.LeakyReLU(negative_slope=0.1),
        'ralu':       RALU(beta=0.9),
        'elu':        nn.ELU(),
        'selu':       nn.SELU(),
        'tanh':       nn.Tanh(),
        'sigmoid':    nn.Sigmoid(),
        'gelu':       nn.GELU(),
        'swish':      nn.SiLU(),
        'mish':       nn.Mish(),
        'hardswish':  nn.Hardswish(inplace=False),
    }
    if name not in mapping:
        print(f"Warning: unknown activation '{name}', using ReLU.")
        return nn.ReLU()
    return mapping[name]


ACT_LABELS = {
    'relu':       'ReLU',
    'leaky_relu': 'Leaky ReLU',
    'ralu':        'RALU',
    'gelu':       'GELU',
    'swish':      'Swish',
    'mish':       'Mish',
    'prelu':      'PReLU',
    'selu':       'SELU',
    'tanh':       'Tanh',
    'hardswish':  'HardSwish',
}

ACT_COLOURS = {
    'relu':       '#E15759',  # Red
    'leaky_relu': '#000000',  # Black
    'ralu':        '#59A14F',  # Green
    'gelu':       '#1D5D9B',  # Navy
    'swish':      '#B07AA1',  # Purple
    'mish':       '#EDC948',  # Yellow
    'prelu':      '#FF9DA7',  # Pink
    'selu':       '#4E79A7',  # Blue
    'tanh':       '#595959',  # Dark Grey
    'hardswish':  '#9C755F',  # Brown
}


# ============================================================================
#  MODELS
# ============================================================================
class ShallowCNN(nn.Module):
    def __init__(self, num_classes=10, input_channels=3, activation='relu'):
        super().__init__()
        self.activation_type = activation
        self.conv1 = nn.Conv2d(input_channels, 32, 3, padding=1)
        self.bn1   = nn.BatchNorm2d(32);  self.act1 = get_activation(activation)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2   = nn.BatchNorm2d(64);  self.act2 = get_activation(activation)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(128); self.act3 = get_activation(activation)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1      = nn.Linear(128, 256); self.act_fc1 = get_activation(activation)
        self.dropout1 = nn.Dropout(0.5)
        self.fc2      = nn.Linear(256, num_classes)
        self.activations = {}

    def forward(self, x, save_activations=False):
        out = self.pool1(self.act1(self.bn1(self.conv1(x))))
        if save_activations: self.activations['block1'] = out.detach()
        out = self.pool2(self.act2(self.bn2(self.conv2(out))))
        if save_activations: self.activations['block2'] = out.detach()
        out = self.pool3(self.act3(self.bn3(self.conv3(out))))
        if save_activations: self.activations['block3'] = out.detach()
        out = self.adaptive_pool(out)
        out = out.view(out.size(0), -1)
        out = self.dropout1(self.act_fc1(self.fc1(out)))
        return self.fc2(out)


class VGG16Model(nn.Module):
    def __init__(self, num_classes=10, input_channels=3, activation='relu'):
        super().__init__()
        self.activation_type = activation

        def _blk(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                get_activation(activation))

        self.block1 = nn.Sequential(_blk(input_channels, 64), _blk(64, 64),   nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(_blk(64, 128),  _blk(128, 128),            nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(_blk(128, 256), _blk(256, 256), _blk(256, 256), nn.MaxPool2d(2, 2))
        self.block4 = nn.Sequential(_blk(256, 512), _blk(512, 512), _blk(512, 512), nn.MaxPool2d(2, 2))
        self.block5 = nn.Sequential(_blk(512, 512), _blk(512, 512), _blk(512, 512), nn.MaxPool2d(2, 2))
        self.adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc1     = nn.Linear(512, 512); self.act_fc1 = get_activation(activation)
        self.drop    = nn.Dropout(0.5)
        self.fc2     = nn.Linear(512, num_classes)
        self.activations = {}

    def forward(self, x, save_activations=False):
        for i, blk in enumerate([self.block1, self.block2, self.block3,
                                  self.block4, self.block5], 1):
            x = blk(x)
            if save_activations: self.activations[f'block{i}'] = x.detach()
        x = self.adaptive_pool(x).view(x.size(0), -1)
        x = self.drop(self.act_fc1(self.fc1(x)))
        return self.fc2(x)


class DenseLayer(nn.Module):
    def __init__(self, in_c, growth, activation):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm2d(in_c), get_activation(activation),
            nn.Conv2d(in_c, 4 * growth, 1, bias=False),
            nn.BatchNorm2d(4 * growth), get_activation(activation),
            nn.Conv2d(4 * growth, growth, 3, padding=1, bias=False))

    def forward(self, x):
        return torch.cat([x, self.net(x)], 1)


class DenseNet121Model(nn.Module):
    def __init__(self, num_classes=10, input_channels=3, activation='relu', growth_rate=32):
        super().__init__()
        self.activation_type = activation
        gr = growth_rate
        self.stem = nn.Sequential(
            nn.Conv2d(input_channels, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), get_activation(activation), nn.MaxPool2d(3, stride=2, padding=1))
        nf = 64
        configs = [6, 12, 24, 16]
        self.dense_blocks, self.transitions = nn.ModuleList(), nn.ModuleList()
        for i, nl in enumerate(configs):
            db = nn.Sequential(*[DenseLayer(nf + j * gr, gr, activation) for j in range(nl)])
            self.dense_blocks.append(db); nf += nl * gr
            if i < len(configs) - 1:
                self.transitions.append(nn.Sequential(
                    nn.BatchNorm2d(nf), get_activation(activation),
                    nn.Conv2d(nf, nf // 2, 1, bias=False), nn.AvgPool2d(2, 2)))
                nf //= 2
        self.final = nn.Sequential(nn.BatchNorm2d(nf), get_activation(activation))
        self.pool  = nn.AdaptiveAvgPool2d((1, 1))
        self.fc    = nn.Linear(nf, num_classes)
        self.activations = {}

    def forward(self, x, save_activations=False):
        x = self.stem(x)
        for i, db in enumerate(self.dense_blocks):
            x = db(x)
            if save_activations: self.activations[f'dense{i+1}'] = x.detach()
            if i < len(self.transitions): x = self.transitions[i](x)
        x = self.pool(self.final(x)).view(x.size(0), -1)
        return self.fc(x)


# ============================================================================
#  MobileNetV3 (Small)
# ============================================================================
class MobileNetV3Model(nn.Module):
    def __init__(self, num_classes=10, input_channels=3, activation='relu'):
        super().__init__()
        self.activation_type = activation
        backbone = tv_models.mobilenet_v3_small(weights=None)

        if input_channels != 3:
            first_conv = backbone.features[0][0]
            backbone.features[0][0] = nn.Conv2d(
                input_channels, first_conv.out_channels,
                kernel_size=first_conv.kernel_size,
                stride=first_conv.stride,
                padding=first_conv.padding,
                bias=False)

        self.features    = backbone.features
        self.avgpool     = backbone.avgpool
        in_features      = backbone.classifier[0].in_features
        hidden_features  = 1024

        self.classifier = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            get_activation(activation),
            nn.Dropout(p=0.2),
            nn.Linear(hidden_features, num_classes),
        )
        self.activations = {}

    def forward(self, x, save_activations=False):
        for i, layer in enumerate(self.features):
            x = layer(x)
            if save_activations and i in (3, 8, 11):
                self.activations[f'features_{i}'] = x.detach()
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# ============================================================================
#  ViT-B/16
# ============================================================================
class ViTB16Model(nn.Module):
    def __init__(self, num_classes=10, input_channels=3, activation='relu',
                 image_size=224):
        super().__init__()
        self.activation_type = activation

        vit = tv_models.vit_b_16(weights=None, image_size=image_size)

        if input_channels != 3:
            old_proj = vit.conv_proj
            vit.conv_proj = nn.Conv2d(
                input_channels, old_proj.out_channels,
                kernel_size=old_proj.kernel_size,
                stride=old_proj.stride)

        for block in vit.encoder.layers:
            if hasattr(block, 'mlp') and len(block.mlp) > 1:
                block.mlp[1] = get_activation(activation)

        in_features     = vit.heads.head.in_features
        vit.heads.head  = nn.Linear(in_features, num_classes)

        self.vit         = vit
        self.activations = {}

    def forward(self, x, save_activations=False):
        out = self.vit(x)
        if save_activations:
            self.activations['patch_embed'] = x.detach()
        return out


# ============================================================================
#  EfficientNet-B0
# ============================================================================
class EfficientNetB0Model(nn.Module):
    """
    Wraps torchvision EfficientNet-B0.
    The MBConv backbone's SiLU activations are preserved (structural to the
    inverted-residual blocks).  The custom activation is injected into the
    final classifier head, consistent with the MobileNetV3 strategy above.
    """
    def __init__(self, num_classes=10, input_channels=3, activation='relu'):
        super().__init__()
        self.activation_type = activation
        backbone = tv_models.efficientnet_b0(weights=None)

        if input_channels != 3:
            first_conv = backbone.features[0][0]
            backbone.features[0][0] = nn.Conv2d(
                input_channels, first_conv.out_channels,
                kernel_size=first_conv.kernel_size,
                stride=first_conv.stride,
                padding=first_conv.padding,
                bias=False)

        self.features = backbone.features
        self.avgpool  = backbone.avgpool
        in_features   = backbone.classifier[1].in_features   # 1280 for B0

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(in_features, num_classes),
        )
        self.activations = {}

    def forward(self, x, save_activations=False):
        for i, layer in enumerate(self.features):
            x = layer(x)
            if save_activations and i in (2, 5, 7):
                self.activations[f'features_{i}'] = x.detach()
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# ============================================================================
#  RESNET-34
# ============================================================================
class BasicBlock(nn.Module):
    """
    BasicBlock for ResNet-34.
    Optimized for RALU activation function: f(x) = x + β·x²/(1+|x|)
    Uses pre-activation architecture (BN → Activation → Conv) for better
    gradient flow with rational activations.
    """
    expansion = 1

    def __init__(self, in_planes, planes, stride=1, activation='relu'):
        super().__init__()
        self.activation_type = activation
        act_fn = get_activation(activation)

        # Pre-activation: BN -> Activation -> Conv
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.act1 = act_fn
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride,
                               padding=1, bias=False)

        self.bn2 = nn.BatchNorm2d(planes)
        self.act2 = act_fn
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1,
                               padding=1, bias=False)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.bn1(x)
        out = self.act1(out)
        out = self.conv1(out)

        out = self.bn2(out)
        out = self.act2(out)
        out = self.conv2(out)

        out += identity
        return out


class ResNet34Model(nn.Module):
    """
    ResNet-34 with flexible activation functions.
    Optimized architecture for RALU: f(x) = x + β·x²/(1+|x|)

    Key design choices for RALU compatibility:
    1. Pre-activation blocks (BN before activation) for stable gradient flow
    2. Learnable parameters in RALU are properly handled
    3. Appropriate weight initialization for rational activations
    4. Batch norm after each convolution before activation
    """
    def __init__(self, num_classes=10, input_channels=3, activation='relu'):
        super().__init__()
        self.activation_type = activation

        # Initial convolution (no activation yet - will be in first block)
        self.conv1 = nn.Conv2d(input_channels, 64, kernel_size=7, stride=2,
                               padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.act1 = get_activation(activation)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet-34 stages
        self.layer1 = self._make_layer(64, 64, 3, stride=1, activation=activation)
        self.layer2 = self._make_layer(64, 128, 4, stride=2, activation=activation)
        self.layer3 = self._make_layer(128, 256, 6, stride=2, activation=activation)
        self.layer4 = self._make_layer(256, 512, 3, stride=2, activation=activation)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * BasicBlock.expansion, num_classes)

        self.activations = {}

        # Initialize weights with appropriate scheme
        self._initialize_weights(activation)

    def _make_layer(self, in_planes, planes, num_blocks, stride, activation):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride_val in strides:
            layers.append(BasicBlock(in_planes, planes, stride_val, activation))
            in_planes = planes * BasicBlock.expansion
        return nn.Sequential(*layers)

    def _initialize_weights(self, activation):
        """Initialize weights."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                if activation.lower() in ['relu', 'leaky_relu', 'ralu', 'raglu']:
                    # Kaiming initialization
                    nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                else:
                    # Xavier for smooth activations like swish, gelu, mish
                    nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x, save_activations=False):
        # Initial stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.act1(x)
        if save_activations:
            self.activations['stem'] = x.detach()
        x = self.maxpool(x)

        # ResNet stages
        x = self.layer1(x)
        if save_activations:
            self.activations['layer1'] = x.detach()

        x = self.layer2(x)
        if save_activations:
            self.activations['layer2'] = x.detach()

        x = self.layer3(x)
        if save_activations:
            self.activations['layer3'] = x.detach()

        x = self.layer4(x)
        if save_activations:
            self.activations['layer4'] = x.detach()

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


# ============================================================================
#  MODEL FACTORY & WEIGHT INIT
# ============================================================================
MODEL_MIN_SIZE = {
    'shallowcnn':    32,
    'vgg16':         32,
    'densenet121':   32,
    'mobilenetv3':   32,
    'efficientnetb0': 32,
    'resnet34':      32,      # ResNet-34 can handle 32x32 with modified stem
    'vitb16':        224,
}


def build_model(model_type, num_classes, input_channels, activation):
    mt = model_type.lower()
    if mt == 'shallowcnn':
        return ShallowCNN(num_classes, input_channels, activation)
    elif mt == 'vgg16':
        return VGG16Model(num_classes, input_channels, activation)
    elif mt == 'densenet121':
        return DenseNet121Model(num_classes, input_channels, activation)
    elif mt == 'mobilenetv3':
        return MobileNetV3Model(num_classes, input_channels, activation)
    elif mt == 'efficientnetb0':
        return EfficientNetB0Model(num_classes, input_channels, activation)
    elif mt == 'resnet34':
        return ResNet34Model(num_classes, input_channels, activation)
    elif mt == 'vitb16':
        img_sz = MODEL_MIN_SIZE['vitb16']
        return ViTB16Model(num_classes, input_channels, activation, image_size=img_sz)
    raise ValueError(f"Unknown model: {model_type}")


def init_weights(m, activation):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        if activation.lower() in ['relu', 'leaky_relu']:
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        else:
            nn.init.xavier_normal_(m.weight)
        if m.bias is not None: nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)


# ============================================================================
#  HELPER: strip torch.compile prefix from state dict keys
# ============================================================================
def strip_compile_prefix(state_dict):
    """Remove '_orig_mod.' prefix added by torch.compile(), if present."""
    return {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}


# ============================================================================
#  DATASET LOADER
# ============================================================================
def _get_image_size():
    return MODEL_MIN_SIZE.get(MODEL_TYPE.lower(), 32)


def load_datasets(dataset_type):
    img_sz = _get_image_size()

    if dataset_type == 'cifar10':
        mean, std = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
        base_tr = [transforms.RandomHorizontalFlip(),
                   transforms.RandomCrop(32, padding=4)]
        base_te = []
        if img_sz > 32:
            base_tr = [transforms.Resize(img_sz)] + base_tr
            base_te = [transforms.Resize(img_sz)]
        tr = transforms.Compose(base_tr + [transforms.ToTensor(), transforms.Normalize(mean, std)])
        te = transforms.Compose(base_te + [transforms.ToTensor(), transforms.Normalize(mean, std)])
        train = dsets.CIFAR10('./data', train=True,  transform=tr, download=True)
        valid = dsets.CIFAR10('./data', train=False, transform=te, download=True)
        test  = dsets.CIFAR10('./data', train=False, transform=te, download=True)
        classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
        return train, valid, test, 10, 3, img_sz, classes, mean, std

    elif dataset_type == 'cifar100':
        mean, std = (0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)
        base_tr = [transforms.RandomHorizontalFlip(),
                   transforms.RandomCrop(32, padding=4)]
        base_te = []
        if img_sz > 32:
            base_tr = [transforms.Resize(img_sz)] + base_tr
            base_te = [transforms.Resize(img_sz)]
        tr = transforms.Compose(base_tr + [transforms.ToTensor(), transforms.Normalize(mean, std)])
        te = transforms.Compose(base_te + [transforms.ToTensor(), transforms.Normalize(mean, std)])
        train = dsets.CIFAR100('./data', train=True,  transform=tr, download=True)
        valid = dsets.CIFAR100('./data', train=False, transform=te, download=True)
        test  = dsets.CIFAR100('./data', train=False, transform=te, download=True)
        return train, valid, test, 100, 3, img_sz, train.classes, mean, std

    elif dataset_type == 'tinyimagenet200':
        _prepare_tinyimagenet(TINYIMAGENET_DIR)
        mean, std = (0.4802, 0.4481, 0.3975), (0.2770, 0.2691, 0.2821)
        native = 64
        if img_sz > native:
            tr = transforms.Compose([
                transforms.Resize(img_sz),
                transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(img_sz, padding=img_sz // 8),
                transforms.ToTensor(), transforms.Normalize(mean, std)])
            te = transforms.Compose([
                transforms.Resize(img_sz),
                transforms.CenterCrop(img_sz),
                transforms.ToTensor(), transforms.Normalize(mean, std)])
        else:
            tr = transforms.Compose([
                transforms.Resize(native), transforms.RandomHorizontalFlip(),
                transforms.RandomCrop(native, padding=8),
                transforms.ToTensor(), transforms.Normalize(mean, std)])
            te = transforms.Compose([
                transforms.Resize(native), transforms.CenterCrop(native),
                transforms.ToTensor(), transforms.Normalize(mean, std)])
        train = dsets.ImageFolder(os.path.join(TINYIMAGENET_DIR, 'train'), transform=tr)
        valid = dsets.ImageFolder(os.path.join(TINYIMAGENET_DIR, 'val'),   transform=te)
        test  = valid
        return train, valid, test, 200, 3, img_sz, train.classes, mean, std

    elif dataset_type == 'custom':
        mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        sz = max(img_sz, 128)
        tr = transforms.Compose([transforms.Resize((sz, sz)), transforms.RandomHorizontalFlip(),
                                  transforms.RandomRotation(10),
                                  transforms.ToTensor(), transforms.Normalize(mean, std)])
        te = transforms.Compose([transforms.Resize((sz, sz)),
                                  transforms.ToTensor(), transforms.Normalize(mean, std)])
        train = dsets.ImageFolder(TRAIN_DIR, transform=tr)
        valid = dsets.ImageFolder(VALID_DIR, transform=te)
        test  = dsets.ImageFolder(TEST_DIR,  transform=te)
        return train, valid, test, len(train.classes), 3, sz, train.classes, mean, std

    raise ValueError(f"Unknown dataset: {dataset_type}")


def _prepare_tinyimagenet(dest):
    import urllib.request, zipfile
    url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
    zp  = '/content/tiny-imagenet-200.zip'
    if not os.path.isdir(dest):
        urllib.request.urlretrieve(url, zp)
        with zipfile.ZipFile(zp, 'r') as zf: zf.extractall('/content')
        os.remove(zp)
    val_img = os.path.join(dest, 'val', 'images')
    val_ann = os.path.join(dest, 'val', 'val_annotations.txt')
    if os.path.isdir(val_img):
        with open(val_ann) as f:
            for line in f:
                p = line.strip().split('\t')
                dst_dir = os.path.join(dest, 'val', p[1])
                os.makedirs(dst_dir, exist_ok=True)
                src = os.path.join(val_img, p[0]); dst = os.path.join(dst_dir, p[0])
                if os.path.isfile(src): shutil.move(src, dst)
        try: os.rmdir(val_img)
        except OSError: pass


# ============================================================================
#  EARLY STOPPING
# ============================================================================
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience  = patience
        self.counter   = 0
        self.best_loss = None
        self.stop      = False

    def __call__(self, val_loss):
        if self.best_loss is None or val_loss < self.best_loss - 1e-4:
            self.best_loss = val_loss; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True

    def reset(self):
        self.counter = 0; self.best_loss = None; self.stop = False


# ============================================================================
#  OPTIMIZER FACTORY
# ============================================================================
def build_optimizer(model, lr, optimizer_type):
    params = model.parameters()
    ot = optimizer_type.lower()
    if ot == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    elif ot == 'adam':
        return torch.optim.Adam(params, lr=lr, weight_decay=5e-4)
    elif ot == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=5e-4)
    elif ot == 'adagrad':
        return torch.optim.Adagrad(params, lr=lr, weight_decay=5e-4)
    elif ot == 'adamax':
        return torch.optim.Adamax(params, lr=lr, weight_decay=5e-4)
    elif ot == 'nadam':
        return torch.optim.NAdam(params, lr=lr, weight_decay=5e-4)
    elif ot == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, weight_decay=5e-4)
    elif ot == 'rprop':
        return torch.optim.Rprop(params, lr=lr)
    elif ot == 'adadelta':
        return torch.optim.Adadelta(params, lr=lr, weight_decay=5e-4)
    else:
        print(f"Warning: unknown optimizer '{optimizer_type}', using Adam.")
        return torch.optim.Adam(params, lr=lr, weight_decay=5e-4)


# ============================================================================
#  TRAINING LOOP  (with AMP)
# ============================================================================
def train_one_activation(activation_name, train_loader, valid_loader,
                          num_classes, input_channels, device, output_dir):
    print(f"\n{'='*60}")
    print(f"  Training with: {ACT_LABELS.get(activation_name, activation_name).upper()}")
    print(f"{'='*60}")

    model = build_model(MODEL_TYPE, num_classes, input_channels, activation_name)
    model.apply(lambda m: init_weights(m, activation_name))
    model = model.to(device)

    if hasattr(torch, 'compile'):
        try:
            model = torch.compile(model)
        except Exception:
            pass

    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, LEARNING_RATE, OPTIMIZER_TYPE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    use_amp  = USE_AMP and device.type == 'cuda'
    scaler   = torch.cuda.amp.GradScaler(enabled=use_amp)

    es = EarlyStopping(EARLY_STOPPING_PATIENCE) if EARLY_STOPPING else None

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc':  [],
        'lr': [],         'epoch_time': [],
    }

    for epoch in range(NUM_EPOCHS):
        t0 = time.time()
        model.train()
        tr_loss = tr_correct = tr_total = 0

        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=use_amp):
                out  = model(images)
                loss = criterion(out, labels)

            if torch.isnan(loss): continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()

            tr_loss    += loss.item() * images.size(0)
            _, pred     = torch.max(out.detach(), 1)
            tr_correct  += (pred == labels).sum().item()
            tr_total    += labels.size(0)

        scheduler.step()

        model.eval()
        v_loss = v_correct = v_total = 0
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
            for images, labels in valid_loader:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                out     = model(images)
                v_loss  += criterion(out, labels).item() * images.size(0)
                _, pred  = torch.max(out, 1)
                v_correct += (pred == labels).sum().item()
                v_total   += labels.size(0)

        t1   = time.time()
        tr_l = tr_loss / tr_total;   tr_a = 100 * tr_correct / tr_total
        va_l = v_loss  / v_total;    va_a = 100 * v_correct  / v_total
        lr   = scheduler.get_last_lr()[0]

        history['train_loss'].append(tr_l); history['val_loss'].append(va_l)
        history['train_acc'].append(tr_a);  history['val_acc'].append(va_a)
        history['lr'].append(lr);           history['epoch_time'].append(t1 - t0)

        print(f"  Epoch {epoch+1:3d}/{NUM_EPOCHS}  "
              f"TrainL={tr_l:.4f}  ValL={va_l:.4f}  "
              f"TrainAcc={tr_a:.2f}%  ValAcc={va_a:.2f}%  "
              f"LR={lr:.6f}  Time={t1-t0:.1f}s")

        if es:
            es(va_l)
            if es.stop:
                print(f"  Early stopping at epoch {epoch+1}.")
                break

    return model, history


# ============================================================================
#  EVALUATION HELPERS
# ============================================================================
def evaluate_model(model, loader, device, num_classes):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    use_amp = USE_AMP and device.type == 'cuda'
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        for images, labels in loader:
            out   = model(images.to(device, non_blocking=True))
            probs = torch.softmax(out, 1).cpu().numpy()
            _, p  = torch.max(out, 1)
            all_probs.append(probs)
            all_preds.extend(p.cpu().numpy())
            all_labels.extend(labels.numpy())
    return (np.array(all_labels),
            np.array(all_preds),
            np.vstack(all_probs))


# ============================================================================
#  UTILITY
# ============================================================================
def denorm(img_tensor, mean, std):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = np.array(std) * img + np.array(mean)
    return np.clip(img, 0, 1)


def make_dir(*parts):
    p = os.path.join(*parts)
    os.makedirs(p, exist_ok=True)
    return p


# ============================================================================
#  PER-ACTIVATION VISUALISATIONS
# ============================================================================

def plot_training_curves(history, activation, out_dir):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    c = ACT_COLOURS.get(activation, '#333333')
    label = ACT_LABELS.get(activation, activation)

    axes[0].plot(epochs, history['train_loss'], color=c, lw=2, label='Train')
    axes[0].plot(epochs, history['val_loss'],   color=c, lw=2, ls='--', label='Val')
    axes[0].set(title='Loss Curve', xlabel='Epoch', ylabel='Loss')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], color=c, lw=2, label='Train')
    axes[1].plot(epochs, history['val_acc'],   color=c, lw=2, ls='--', label='Val')
    axes[1].set(title='Accuracy Curve', xlabel='Epoch', ylabel='Accuracy (%)')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs, history['lr'], color=c, lw=2)
    axes[2].set(title='Learning Rate Schedule', xlabel='Epoch', ylabel='LR')
    axes[2].grid(True, alpha=0.3)

    fig.suptitle(f'Training Curves — {label}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(out_dir, f'{activation}_training_curves.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_confusion_matrix_single(labels, preds, class_names, activation, out_dir, split='test'):
    cm    = confusion_matrix(labels, preds)
    n     = len(class_names)
    fig_sz = max(8, n // 4)
    plt.figure(figsize=(fig_sz, fig_sz - 1))
    tick_labels = class_names if n <= 30 else range(n)
    sns.heatmap(cm, annot=(n <= 20), fmt='d', cmap='Blues', square=True,
                xticklabels=tick_labels, yticklabels=tick_labels)
    plt.xlabel('Predicted'); plt.ylabel('True')
    label = ACT_LABELS.get(activation, activation)
    plt.title(f'Confusion Matrix [{split.upper()}] — {label}', fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=6 if n > 20 else 9)
    plt.yticks(rotation=0,  fontsize=6 if n > 20 else 9)
    plt.tight_layout()
    path = os.path.join(out_dir, f'{activation}_{split}_confusion_matrix.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_roc_pr(labels, probs, num_classes, activation, out_dir, split='test'):
    labels_bin = label_binarize(labels, classes=range(num_classes))
    fpr, tpr, roc_auc = {}, {}, {}
    prec, rec, ap = {}, {}, {}

    for i in range(num_classes):
        fpr[i], tpr[i], _ = roc_curve(labels_bin[:, i], probs[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        prec[i], rec[i], _ = precision_recall_curve(labels_bin[:, i], probs[:, i])
        ap[i] = average_precision_score(labels_bin[:, i], probs[:, i])

    fpr['micro'], tpr['micro'], _ = roc_curve(labels_bin.ravel(), probs.ravel())
    roc_auc['micro'] = auc(fpr['micro'], tpr['micro'])
    prec['micro'], rec['micro'], _ = precision_recall_curve(labels_bin.ravel(), probs.ravel())
    ap['micro'] = average_precision_score(labels_bin, probs, average='micro')

    c       = ACT_COLOURS.get(activation, '#333333')
    colours = plt.cm.tab20(np.linspace(0, 1, num_classes))
    label   = ACT_LABELS.get(activation, activation)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    for i, col in zip(range(num_classes), colours):
        ax1.plot(fpr[i], tpr[i], color=col, lw=1.0, alpha=0.6,
                 label=f'C{i} (AUC={roc_auc[i]:.2f})')
    ax1.plot(fpr['micro'], tpr['micro'], color=c, lw=2.5, ls='--',
             label=f'Micro-avg (AUC={roc_auc["micro"]:.2f})')
    ax1.plot([0, 1], [0, 1], 'k--', lw=1)
    ax1.set(xlim=[0, 1], ylim=[0, 1.05], xlabel='FPR', ylabel='TPR',
            title=f'ROC Curves [{split}] — {label}')
    ax1.legend(fontsize=5 if num_classes > 20 else 8, loc='lower right')
    ax1.grid(True, alpha=0.3)

    for i, col in zip(range(num_classes), colours):
        ax2.plot(rec[i], prec[i], color=col, lw=1.0, alpha=0.6,
                 label=f'C{i} (AP={ap[i]:.2f})')
    ax2.plot(rec['micro'], prec['micro'], color=c, lw=2.5, ls='--',
             label=f'Micro-avg (AP={ap["micro"]:.2f})')
    ax2.set(xlim=[0, 1], ylim=[0, 1.05], xlabel='Recall', ylabel='Precision',
            title=f'PR Curves [{split}] — {label}')
    ax2.legend(fontsize=5 if num_classes > 20 else 8, loc='lower left')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(out_dir, f'{activation}_{split}_roc_pr.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_predictions(model, loader, device, class_names, activation, out_dir,
                     mean, std, n=12):
    model.eval()
    images, labels = next(iter(loader))
    imgs   = images[:n].to(device)
    use_amp = USE_AMP and device.type == 'cuda'
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        out   = model(imgs)
        probs = torch.softmax(out, 1).cpu().numpy()
        _, p  = torch.max(out, 1)
    p = p.cpu().numpy()

    cols = 4; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    axes = axes.ravel()
    label = ACT_LABELS.get(activation, activation)

    for idx in range(n):
        img  = denorm(images[idx], mean, std)
        true = labels[idx].item()
        pred = p[idx]
        conf = probs[idx][pred] * 100
        col  = 'green' if true == pred else 'red'
        axes[idx].imshow(img)
        axes[idx].set_title(
            f'T:{class_names[true]}\nP:{class_names[pred]} ({conf:.1f}%)',
            fontsize=8, color=col, fontweight='bold')
        axes[idx].axis('off')
    for idx in range(n, len(axes)): axes[idx].axis('off')

    fig.suptitle(f'Prediction Samples — {label}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(out_dir, f'{activation}_predictions.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_failed_predictions(model, loader, device, class_names, activation,
                             out_dir, mean, std, max_fail=20):
    model.eval()
    fail_imgs, fail_true, fail_pred, fail_probs = [], [], [], []
    use_amp = USE_AMP and device.type == 'cuda'
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        for imgs, lbls in loader:
            out   = model(imgs.to(device, non_blocking=True))
            probs = torch.softmax(out, 1)
            _, p  = torch.max(out, 1)
            mask  = (p.cpu() != lbls)
            for i in range(len(lbls)):
                if mask[i]:
                    fail_imgs.append(imgs[i]); fail_true.append(lbls[i].item())
                    fail_pred.append(p[i].cpu().item())
                    fail_probs.append(probs[i].cpu().numpy())
                if len(fail_imgs) >= max_fail: break
            if len(fail_imgs) >= max_fail: break

    n = len(fail_imgs)
    if n == 0:
        print(f"  No failures for {activation}!"); return

    label = ACT_LABELS.get(activation, activation)
    cols  = 5; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3.5))
    axes = axes.ravel()

    for idx in range(n):
        img  = denorm(fail_imgs[idx], mean, std)
        conf = fail_probs[idx][fail_pred[idx]] * 100
        axes[idx].imshow(img)
        axes[idx].set_title(
            f'T:{class_names[fail_true[idx]]}\n'
            f'P:{class_names[fail_pred[idx]]} ({conf:.1f}%)',
            fontsize=7, color='red', fontweight='bold')
        axes[idx].axis('off')
    for idx in range(n, len(axes)): axes[idx].axis('off')

    fig.suptitle(f'Failed Predictions ({n}) — {label}', fontsize=12,
                 fontweight='bold', color='darkred')
    plt.tight_layout()
    path = os.path.join(out_dir, f'{activation}_failed_predictions.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_feature_maps(model, loader, device, class_names, activation,
                      out_dir, mean, std, n_samples=2):
    model.eval()
    images, labels = next(iter(loader))
    imgs = images[:n_samples].to(device)
    use_amp = USE_AMP and device.type == 'cuda'
    with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
        _ = model(imgs, save_activations=True)

    block_keys = list(model.activations.keys())
    n_blocks   = len(block_keys)
    label      = ACT_LABELS.get(activation, activation)

    for si in range(n_samples):
        n_rows = n_blocks + 1
        fig    = plt.figure(figsize=(20, 3 * n_rows))
        gs     = fig.add_gridspec(n_rows, 8, hspace=0.5, wspace=0.3)

        ax = fig.add_subplot(gs[0, :4])
        ax.imshow(denorm(images[si], mean, std))
        ax.set_title(f'Original | Label: {class_names[labels[si].item()]}',
                     fontweight='bold', fontsize=11)
        ax.axis('off')

        for bi, bk in enumerate(block_keys):
            act_maps = model.activations[bk][si].cpu()
            for fi in range(min(8, act_maps.shape[0])):
                ax = fig.add_subplot(gs[bi + 1, fi])
                ax.imshow(act_maps[fi], cmap='viridis')
                if fi == 0:
                    ax.set_ylabel(bk, fontsize=8, fontweight='bold',
                                  rotation=0, ha='right', va='center')
                ax.set_title(f'F{fi+1}', fontsize=7)
                ax.axis('off')

        fig.suptitle(f'Feature Maps — {label} — Sample {si+1}',
                     fontsize=13, fontweight='bold')
        path = os.path.join(out_dir, f'{activation}_feature_maps_s{si+1}.png')
        plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
        print(f"  Saved: {path}")


def plot_activation_shapes(activations_list, out_dir):
    x   = torch.linspace(-4, 4, 400)
    n   = len(activations_list)
    cols = 3; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, rows * 4))
    axes = np.array(axes).ravel()
    for ax, act in zip(axes, activations_list):
        fn = get_activation(act)
        fn.eval()
        with torch.no_grad():
            y = fn(x).numpy()
        c = ACT_COLOURS.get(act, '#333')
        ax.plot(x.numpy(), y, color=c, lw=2.5)
        ax.axhline(0, color='gray', lw=0.8, ls='--')
        ax.axvline(0, color='gray', lw=0.8, ls='--')
        ax.set_title(ACT_LABELS.get(act, act), fontsize=13, fontweight='bold', color=c)
        ax.set_xlabel('x'); ax.set_ylabel('f(x)')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-4, 4)
    for ax in axes[n:]: ax.axis('off')
    plt.suptitle('Activation Function Shapes', fontsize=15, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(out_dir, 'activation_shapes.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


# ============================================================================
#  CROSS-ACTIVATION COMPARISON VISUALISATIONS
# ============================================================================

def plot_comparison_training_curves(all_history, out_dir):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for act, hist in all_history.items():
        c  = ACT_COLOURS.get(act, '#333')
        lb = ACT_LABELS.get(act, act)
        ep = range(1, len(hist['val_loss']) + 1)
        axes[0].plot(ep, hist['val_loss'], color=c, lw=2, label=lb)
        axes[1].plot(ep, hist['val_acc'],  color=c, lw=2, label=lb)

    axes[0].set(title='Validation Loss', xlabel='Epoch', ylabel='Loss')
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)
    axes[1].set(title='Validation Accuracy (%)', xlabel='Epoch', ylabel='Accuracy (%)')
    axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

    plt.suptitle('Activation Comparison — Training Dynamics', fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_training_curves.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_comparison_bar(metrics_dict, metric_key, title, ylabel, out_dir, fname):
    acts   = list(metrics_dict.keys())
    vals   = [metrics_dict[a][metric_key] for a in acts]
    colors = [ACT_COLOURS.get(a, '#333') for a in acts]
    labels = [ACT_LABELS.get(a, a) for a in acts]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=0.8, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set(title=title, ylabel=ylabel)
    ax.set_ylim(0, max(vals) * 1.15)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(out_dir, fname)
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_comparison_heatmap(metrics_dict, class_names, out_dir):
    acts      = list(metrics_dict.keys())
    act_lbls  = [ACT_LABELS.get(a, a) for a in acts]
    n_cls     = len(class_names)
    show_cls  = class_names[:30]
    f1_matrix = np.array([metrics_dict[a]['per_class_f1'][:30] for a in acts])

    fig, ax = plt.subplots(figsize=(max(12, n_cls // 2), max(4, len(acts) * 1.2)))
    sns.heatmap(f1_matrix, annot=(n_cls <= 20), fmt='.2f', cmap='YlOrRd',
                xticklabels=show_cls, yticklabels=act_lbls,
                ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title('Per-Class F1 Score Heatmap (Activations × Classes)',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Activation')
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_per_class_f1_heatmap.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_radar(metrics_dict, out_dir):
    metric_keys = ['test_acc', 'macro_f1', 'micro_roc_auc', 'micro_ap', 'convergence_speed']
    metric_lbls = ['Test Acc', 'Macro F1', 'ROC-AUC', 'Avg Prec', 'Conv Speed']

    acts   = list(metrics_dict.keys())
    n      = len(metric_keys)
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    for act in acts:
        vals = [metrics_dict[act][k] for k in metric_keys]
        vals[0] = vals[0] / 100.0
        vals += vals[:1]
        c  = ACT_COLOURS.get(act, '#333')
        lb = ACT_LABELS.get(act, act)
        ax.plot(angles, vals, color=c, lw=2, label=lb)
        ax.fill(angles, vals, color=c, alpha=0.08)

    ax.set_thetagrids(np.degrees(angles[:-1]), metric_lbls, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_title('Activation Comparison — Radar Chart', fontsize=13,
                 fontweight='bold', pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_radar.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_comparison_confusion_heatmap(all_cm, class_names, out_dir):
    # FIX: squeeze=False ensures axes is always a 2-D array even with 1 activation
    n_acts = len(all_cm)
    fig, axes = plt.subplots(1, n_acts, figsize=(5 * n_acts, 5), squeeze=False)
    axes = axes.ravel()
    for ax, (act, cm) in zip(axes, all_cm.items()):
        n = len(class_names)
        tick_labels = class_names if n <= 15 else range(n)
        sns.heatmap(cm / cm.sum(axis=1, keepdims=True), cmap='Blues',
                    xticklabels=tick_labels, yticklabels=tick_labels,
                    ax=ax, vmin=0, vmax=1, annot=(n <= 10), fmt='.2f',
                    cbar=False, linewidths=0.3)
        ax.set_title(ACT_LABELS.get(act, act), fontsize=10, fontweight='bold')
        ax.set_xlabel('Pred'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=45, labelsize=6)
        ax.tick_params(axis='y', rotation=0,  labelsize=6)
    plt.suptitle('Normalised Confusion Matrices — All Activations',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_confusion_matrices.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_comparison_roc_overlay(all_micro_roc, out_dir):
    fig, ax = plt.subplots(figsize=(8, 7))
    for act, (fpr, tpr, roc_auc) in all_micro_roc.items():
        c  = ACT_COLOURS.get(act, '#333')
        lb = ACT_LABELS.get(act, act)
        ax.plot(fpr, tpr, color=c, lw=2, label=f'{lb} (AUC={roc_auc:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set(xlim=[0, 1], ylim=[0, 1.05], xlabel='FPR', ylabel='TPR',
           title='Micro-avg ROC Curves — All Activations')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_roc_overlay.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_epoch_time_boxplot(all_history, out_dir):
    fig, ax = plt.subplots(figsize=(10, 5))
    data   = [all_history[a]['epoch_time'] for a in all_history]
    labels = [ACT_LABELS.get(a, a) for a in all_history]
    colors = [ACT_COLOURS.get(a, '#333') for a in all_history]
    bp = ax.boxplot(data, patch_artist=True, labels=labels)
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c); patch.set_alpha(0.7)
    ax.set(title='Per-Epoch Training Time', ylabel='Seconds', xlabel='Activation')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_epoch_time.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_learning_curve_all(all_history, out_dir):
    n = len(all_history)
    cols = 3; rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
    axes = axes.ravel()
    for ax, (act, hist) in zip(axes, all_history.items()):
        c  = ACT_COLOURS.get(act, '#333')
        lb = ACT_LABELS.get(act, act)
        ep = range(1, len(hist['train_loss']) + 1)
        ax2 = ax.twinx()
        ax.plot(ep,  hist['train_loss'], color=c,     lw=2, ls='-',  label='Train Loss')
        ax.plot(ep,  hist['val_loss'],   color=c,     lw=2, ls='--', label='Val Loss', alpha=0.7)
        ax2.plot(ep, hist['train_acc'],  color='gray',  lw=1.5, ls='-',  label='Train Acc')
        ax2.plot(ep, hist['val_acc'],    color='black', lw=1.5, ls='--', label='Val Acc')
        ax.set_title(lb, fontweight='bold', color=c)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss', color=c)
        ax2.set_ylabel('Accuracy (%)')
        ax.grid(True, alpha=0.2)
    for ax in axes[n:]: ax.axis('off')
    plt.suptitle('Individual Training Dynamics — All Activations',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_individual_curves.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


def plot_final_metric_table(metrics_dict, out_dir):
    acts = list(metrics_dict.keys())
    cols = ['Activation', 'Test Acc', 'Val Acc', 'Macro F1', 'Micro ROC-AUC', 'Micro AP']
    rows = []
    for act in acts:
        m = metrics_dict[act]
        rows.append([
            ACT_LABELS.get(act, act),
            f"{m['test_acc']:.2f}%",
            f"{m['val_acc']:.2f}%",
            f"{m['macro_f1']:.4f}",
            f"{m['micro_roc_auc']:.4f}",
            f"{m['micro_ap']:.4f}",
        ])

    fig, ax = plt.subplots(figsize=(14, 1 + 0.5 * len(rows)))
    ax.axis('off')
    tbl = ax.table(cellText=rows, colLabels=cols, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.8)

    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor('#2C3E50')
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    best_acc_idx = np.argmax([metrics_dict[a]['test_acc'] for a in acts])
    for j in range(len(cols)):
        tbl[(best_acc_idx + 1, j)].set_facecolor('#D5F5E3')
    for i in range(1, len(rows) + 1):
        for j in range(len(cols)):
            if i != best_acc_idx + 1 and i % 2 == 0:
                tbl[(i, j)].set_facecolor('#F2F3F4')

    plt.title('Final Performance Summary (Green = Best Test Acc)',
              fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    path = os.path.join(out_dir, 'comparison_summary_table.png')
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {path}")


# ============================================================================
#  CROSS-VALIDATION RUNNER
# ============================================================================

def run_cv_comparison(train_ds, test_loader, num_classes, input_channels,
                      class_names, device, cv_dir, dmean, dstd):
    import warnings
    warnings.filterwarnings("ignore", category=UserWarning)

    print("\n" + "="*80)
    print(f"STARTING {NUMBER_OF_FOLDS}-FOLD CV  |  MODEL:{MODEL_TYPE.upper()}"
          f"  |  DATASET:{DATASET_TYPE.upper()}"
          f"  |  OPTIMIZER:{OPTIMIZER_TYPE.upper()}"
          f"  |  EPOCHS:{CV_EPOCHS}  |  LR:{CV_LEARNING_RATE}")
    print("="*80)

    all_targets = np.array([t for _, t in train_ds])
    skf = StratifiedKFold(n_splits=NUMBER_OF_FOLDS, shuffle=True, random_state=42)

    cv_results = {act: {
        'fold_accuracies':        [],
        'fold_losses':            [],
        'fold_aucs':              [],
        'fold_f1s':               [],
        'fold_precisions':        [],
        'fold_recalls':           [],
        'fold_histories':         [],
        'fold_models':            [],
        'fold_cms':               [],
        'fold_probs':             [],
        'fold_train_times':       [],
        'fold_train_accuracies':  [],
        'fold_train_losses':      [],
        'fold_test_accuracies':   [],
        'fold_test_losses':       [],
        'fold_test_f1s':          [],
        'fold_test_aucs':         [],
    } for act in CV_ACTIVATION_FUNCTIONS}

    criterion = nn.CrossEntropyLoss()
    use_amp   = USE_AMP and device.type == 'cuda'

    for fold, (train_idx, val_idx) in enumerate(
            skf.split(np.zeros(len(all_targets)), all_targets), 1):

        print(f"\n{'-'*60}  FOLD {fold}/{NUMBER_OF_FOLDS}  {'-'*60}")

        tr_sub  = torch.utils.data.Subset(train_ds, train_idx)
        val_sub = torch.utils.data.Subset(train_ds, val_idx)
        tr_ldr  = torch.utils.data.DataLoader(
            tr_sub,  batch_size=BATCH_SIZE, shuffle=True,
            num_workers=2, pin_memory=True, persistent_workers=True)
        va_ldr  = torch.utils.data.DataLoader(
            val_sub, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True, persistent_workers=True)

        for act in CV_ACTIVATION_FUNCTIONS:
            print(f"\n  => {ACT_LABELS.get(act, act).upper()}  (Fold {fold})")

            model = build_model(MODEL_TYPE, num_classes, input_channels, act)
            model.apply(lambda m: init_weights(m, act))
            model = model.to(device)

            if hasattr(torch, 'compile'):
                try:
                    model = torch.compile(model)
                except Exception:
                    pass

            optimizer = build_optimizer(model, CV_LEARNING_RATE, OPTIMIZER_TYPE)
            scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)

            history = {'train_loss': [], 'train_acc': [],
                       'val_loss':   [], 'val_acc':   [],
                       'val_auc':    [], 'val_f1':    []}
            best_val_acc     = 0.0
            best_model_state = None
            fold_start       = time.time()

            for epoch in range(CV_EPOCHS):
                # ---- Train ----
                model.train()
                t_loss = t_correct = t_total = 0
                for imgs, lbls in tr_ldr:
                    imgs, lbls = (imgs.to(device, non_blocking=True),
                                  lbls.to(device, non_blocking=True))
                    optimizer.zero_grad(set_to_none=True)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        out  = model(imgs)
                        loss = criterion(out, lbls)
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()

                    t_loss    += loss.item()
                    _, p       = torch.max(out.detach(), 1)
                    t_correct += (p == lbls).sum().item()
                    t_total   += lbls.size(0)

                avg_tl = t_loss / len(tr_ldr)
                avg_ta = t_correct / t_total

                # ---- Validate ----
                model.eval()
                v_loss = 0
                v_preds, v_labels, v_probs = [], [], []
                with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
                    for imgs, lbls in va_ldr:
                        imgs, lbls = (imgs.to(device, non_blocking=True),
                                      lbls.to(device, non_blocking=True))
                        out    = model(imgs)
                        v_loss += criterion(out, lbls).item()
                        prb    = torch.softmax(out, 1)
                        _, p   = torch.max(out, 1)
                        v_preds.extend(p.cpu().numpy())
                        v_labels.extend(lbls.cpu().numpy())
                        v_probs.extend(prb.cpu().numpy())

                avg_vl  = v_loss / len(va_ldr)
                val_acc = accuracy_score(v_labels, v_preds)
                try:
                    from sklearn.metrics import roc_auc_score
                    val_auc = roc_auc_score(v_labels, v_probs,
                                            multi_class='ovr', average='macro') \
                              if num_classes > 2 else \
                              roc_auc_score(v_labels, np.array(v_probs)[:, 1])
                except: val_auc = 0.0
                try:
                    val_f1   = f1_score(v_labels, v_preds, average='macro', zero_division=0)
                    val_prec = precision_score(v_labels, v_preds, average='macro', zero_division=0)
                    val_rec  = recall_score(v_labels, v_preds, average='macro', zero_division=0)
                except: val_f1 = val_prec = val_rec = 0.0

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_model_state = {
                        'state_dict': strip_compile_prefix(model.state_dict()),
                        'fold': fold, 'activation': act, 'val_acc': val_acc}

                if (epoch + 1) % max(1, CV_EPOCHS // 5) == 0 or epoch == CV_EPOCHS - 1:
                    print(f"     Ep {epoch+1:3d}/{CV_EPOCHS} | "
                          f"TrLoss:{avg_tl:.4f} TrAcc:{avg_ta*100:.1f}% | "
                          f"VaLoss:{avg_vl:.4f} VaAcc:{val_acc*100:.1f}% "
                          f"AUC:{val_auc:.3f} F1:{val_f1:.3f}")

                history['train_loss'].append(avg_tl); history['train_acc'].append(avg_ta)
                history['val_loss'].append(avg_vl);   history['val_acc'].append(val_acc)
                history['val_auc'].append(val_auc);   history['val_f1'].append(val_f1)

            fold_elapsed = time.time() - fold_start
            print(f"     Time: {fold_elapsed:.1f}s  ({fold_elapsed/CV_EPOCHS:.2f}s/ep)")

            # ---- Test set inference ----
            model.eval()
            te_loss = 0
            te_preds, te_labels, te_probs = [], [], []
            with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
                for imgs, lbls in test_loader:
                    imgs, lbls = (imgs.to(device, non_blocking=True),
                                  lbls.to(device, non_blocking=True))
                    out    = model(imgs)
                    te_loss += criterion(out, lbls).item()
                    prb    = torch.softmax(out, 1)
                    _, p   = torch.max(out, 1)
                    te_preds.extend(p.cpu().numpy())
                    te_labels.extend(lbls.cpu().numpy())
                    te_probs.extend(prb.cpu().numpy())

            avg_tel  = te_loss / len(test_loader)
            test_acc = accuracy_score(te_labels, te_preds)
            try:
                from sklearn.metrics import roc_auc_score
                test_auc = roc_auc_score(te_labels, te_probs,
                                         multi_class='ovr', average='macro') \
                           if num_classes > 2 else \
                           roc_auc_score(te_labels, np.array(te_probs)[:, 1])
            except: test_auc = 0.0
            try:    test_f1 = f1_score(te_labels, te_preds, average='macro', zero_division=0)
            except: test_f1 = 0.0
            n_fail = np.sum(np.array(te_preds) != np.array(te_labels))
            print(f"     [TEST] Acc:{test_acc*100:.2f}%  Loss:{avg_tel:.4f}  "
                  f"AUC:{test_auc:.3f}  F1:{test_f1:.3f}  Fail:{n_fail}")

            cm = confusion_matrix(v_labels, v_preds, labels=list(range(num_classes)))

            r = cv_results[act]
            r['fold_accuracies'].append(best_val_acc)
            r['fold_losses'].append(avg_vl)
            r['fold_aucs'].append(val_auc)
            r['fold_f1s'].append(val_f1)
            r['fold_precisions'].append(val_prec)
            r['fold_recalls'].append(val_rec)
            r['fold_histories'].append(history)
            r['fold_cms'].append(cm)
            r['fold_probs'].append((np.array(v_labels), np.array(v_probs)))
            r['fold_train_times'].append(fold_elapsed)
            r['fold_train_accuracies'].append(history['train_acc'][-1])
            r['fold_train_losses'].append(history['train_loss'][-1])
            r['fold_test_accuracies'].append(test_acc)
            r['fold_test_losses'].append(avg_tel)
            r['fold_test_f1s'].append(test_f1)
            r['fold_test_aucs'].append(test_auc)
            if best_model_state is not None:
                r['fold_models'].append(best_model_state)

            del model, optimizer, scaler
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()

        print(f"\nFold {fold} complete.")

    # ---- Aggregate ----
    rows = {}
    for act in CV_ACTIVATION_FUNCTIONS:
        r = cv_results[act]
        def s(key): return np.array(r[key])
        rows[act] = {
            'val_accuracy_mean':      s('fold_accuracies').mean(),
            'val_accuracy_std':       s('fold_accuracies').std(),
            'val_loss_mean':          s('fold_losses').mean(),
            'val_loss_std':           s('fold_losses').std(),
            'val_auc_mean':           s('fold_aucs').mean(),
            'val_auc_std':            s('fold_aucs').std(),
            'val_f1_mean':            s('fold_f1s').mean(),
            'val_f1_std':             s('fold_f1s').std(),
            'val_precision_mean':     s('fold_precisions').mean(),
            'val_precision_std':      s('fold_precisions').std(),
            'val_recall_mean':        s('fold_recalls').mean(),
            'val_recall_std':         s('fold_recalls').std(),
            'train_accuracy_mean':    s('fold_train_accuracies').mean(),
            'train_accuracy_std':     s('fold_train_accuracies').std(),
            'train_loss_mean':        s('fold_train_losses').mean(),
            'train_loss_std':         s('fold_train_losses').std(),
            'test_accuracy_mean':     s('fold_test_accuracies').mean(),
            'test_accuracy_std':      s('fold_test_accuracies').std(),
            'test_loss_mean':         s('fold_test_losses').mean(),
            'test_loss_std':          s('fold_test_losses').std(),
            'test_f1_mean':           s('fold_test_f1s').mean(),
            'test_f1_std':            s('fold_test_f1s').std(),
            'test_auc_mean':          s('fold_test_aucs').mean(),
            'test_auc_std':           s('fold_test_aucs').std(),
            'train_time_mean_s':      s('fold_train_times').mean(),
            'train_time_std_s':       s('fold_train_times').std(),
            'train_time_per_epoch_s': s('fold_train_times').mean() / CV_EPOCHS,
        }

    summary_df = pd.DataFrame(rows).T.round(4)
    print("\nCV Summary (mean ± std):\n", summary_df.to_string())
    csv_path = os.path.join(cv_dir, 'cv_summary.csv')
    summary_df.to_csv(csv_path)
    print(f"Saved: {csv_path}")

    return cv_results, summary_df


# ============================================================================
#  CV VISUALISATION FUNCTIONS
# ============================================================================

def _cv_colours(acts):
    base = plt.cm.tab10(np.linspace(0, 1, len(acts)))
    return [ACT_COLOURS.get(a, base[i]) for i, a in enumerate(acts)]


def cv_plot_radar(summary_df, acts, cv_dir):
    categories = ['Val Acc', 'Val AUC', 'Val F1', 'Val Prec', 'Val Rec']
    keys       = ['val_accuracy_mean', 'val_auc_mean', 'val_f1_mean',
                  'val_precision_mean', 'val_recall_mean']
    N      = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)] + [0]
    colours = _cv_colours(acts)

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    ax.set_theta_offset(pi / 2); ax.set_theta_direction(-1)
    plt.xticks(angles[:-1], categories, size=12)
    plt.yticks([0.2, 0.4, 0.6, 0.8], ['0.2', '0.4', '0.6', '0.8'], color='grey', size=10)
    plt.ylim(0, 1)
    for c, act in zip(colours, acts):
        vals = [summary_df.loc[act, k] for k in keys] + [summary_df.loc[act, keys[0]]]
        ax.plot(angles, vals, lw=2, label=ACT_LABELS.get(act, act), color=c)
        ax.fill(angles, vals, alpha=0.08, color=c)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
    ax.set_title(f'CV Mean Performance Radar ({NUMBER_OF_FOLDS}-Fold)',
                 size=13, fontweight='bold', pad=25)
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_radar.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_bar(summary_df, acts, mean_key, std_key, title, ylabel, colour_face,
                colour_edge, cv_dir, fname):
    x      = np.arange(len(acts))
    means  = [summary_df.loc[a, mean_key] for a in acts]
    stds   = [summary_df.loc[a, std_key]  for a in acts]
    labels = [ACT_LABELS.get(a, a) for a in acts]
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(x, means, 0.6, yerr=stds, capsize=5,
                  color=colour_face, edgecolor=colour_edge, alpha=0.85)
    for bar, v, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + s + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel); ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    p = os.path.join(cv_dir, fname)
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_training_time(summary_df, acts, cv_dir):
    x      = np.arange(len(acts))
    means  = [summary_df.loc[a, 'train_time_mean_s'] for a in acts]
    stds   = [summary_df.loc[a, 'train_time_std_s']  for a in acts]
    labels = [ACT_LABELS.get(a, a) for a in acts]
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(x, means, 0.6, yerr=stds, capsize=5,
                  color='lightsteelblue', edgecolor='steelblue', alpha=0.85)
    for bar, act in zip(bars, acts):
        per_ep = summary_df.loc[act, 'train_time_per_epoch_s']
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{per_ep:.2f}s/ep', ha='center', va='bottom', fontsize=8)
    ax.set_title('Mean Training Time per Fold (±1 std)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Seconds'); ax.set_xticks(x); ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_training_time.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_fold_heatmap(cv_results, acts, metric_key, title, cmap,
                         cbar_label, cv_dir, fname, scale=1.0):
    matrix = np.array([np.array(cv_results[a][metric_key]) * scale for a in acts])
    fig, ax = plt.subplots(figsize=(max(8, NUMBER_OF_FOLDS * 1.5), max(4, len(acts) * 0.8)))
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap=cmap,
                xticklabels=[f'Fold {i+1}' for i in range(NUMBER_OF_FOLDS)],
                yticklabels=[ACT_LABELS.get(a, a) for a in acts],
                cbar_kws={'label': cbar_label}, ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Fold'); ax.set_ylabel('Activation')
    plt.tight_layout()
    p = os.path.join(cv_dir, fname)
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_learning_curves(cv_results, acts, cv_dir):
    colours  = _cv_colours(acts)
    fig, axes = plt.subplots(3, 1, figsize=(13, 14))

    for c, act in zip(colours, acts):
        hists = cv_results[act]['fold_histories']
        lb    = ACT_LABELS.get(act, act)
        for key, ax in [('train_loss', axes[0]),
                         ('val_acc',   axes[1]),
                         ('train_acc', axes[2])]:
            data = [h[key] for h in hists]
            max_len = max(len(row) for row in data)
            padded  = np.array([np.pad(row, (0, max_len - len(row)),
                                       mode='edge') for row in data])
            ep = np.arange(1, max_len + 1)
            m, s = padded.mean(0), padded.std(0)
            ax.plot(ep, m, lw=2, label=lb, color=c)
            ax.fill_between(ep, m - s, m + s, alpha=0.12, color=c)

    cfg = [(axes[0], f'Avg Train Loss ({NUMBER_OF_FOLDS}-Fold)',    'Loss',     None),
           (axes[1], f'Avg Val Accuracy ({NUMBER_OF_FOLDS}-Fold)',  'Accuracy', (0, 1)),
           (axes[2], f'Avg Train Accuracy ({NUMBER_OF_FOLDS}-Fold)', 'Accuracy', (0, 1))]
    for ax, title, ylabel, ylim in cfg:
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        if ylim: ax.set_ylim(*ylim)
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_learning_curves.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_individual_fold_curves(cv_results, acts, cv_dir):
    colours  = _cv_colours(acts)
    fold_cls = plt.cm.Set2(np.linspace(0, 1, NUMBER_OF_FOLDS))

    for c, act in zip(colours, acts):
        hists = cv_results[act]['fold_histories']
        lb    = ACT_LABELS.get(act, act)
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))

        for fi, h in enumerate(hists):
            ep = np.arange(1, len(h['train_loss']) + 1)
            axes[0].plot(ep, h['train_loss'], color=fold_cls[fi], lw=1.5,
                         label=f'Fold {fi+1}', alpha=0.85)
            axes[1].plot(ep, h['val_acc'],    color=fold_cls[fi], lw=1.5, alpha=0.85)
            axes[2].plot(ep, h['val_f1'],     color=fold_cls[fi], lw=1.5, alpha=0.85)

        titles = ['Train Loss (each fold)', 'Val Accuracy (each fold)', 'Val F1 (each fold)']
        ylbls  = ['Loss', 'Accuracy', 'F1']
        for ax, t, yl in zip(axes, titles, ylbls):
            ax.set_title(t, fontsize=11)
            ax.set_xlabel('Epoch'); ax.set_ylabel(yl)
            ax.grid(True, alpha=0.3)
        axes[0].legend(fontsize=8)

        fig.suptitle(f'Per-Fold Curves — {lb}', fontsize=13, fontweight='bold')
        plt.tight_layout()
        p = os.path.join(cv_dir, f'cv_{act}_fold_curves.png')
        plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
        print(f"  Saved: {p}")


def cv_plot_avg_confusion_matrices(cv_results, acts, class_names, cv_dir):
    n    = len(acts)
    cols = min(3, n); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).ravel()
    nc   = len(class_names)
    tick_l = class_names if nc <= 10 else range(nc)
    for ax, act in zip(axes, acts):
        avg_cm = np.mean(cv_results[act]['fold_cms'], axis=0)
        sns.heatmap(avg_cm, annot=(nc <= 10), fmt='.1f', cmap='Blues', ax=ax, cbar=False,
                    xticklabels=tick_l, yticklabels=tick_l, linewidths=0.3)
        ax.set_title(ACT_LABELS.get(act, act), fontweight='bold')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=45, labelsize=6 if nc > 10 else 9)
        ax.tick_params(axis='y', rotation=0,  labelsize=6 if nc > 10 else 9)
    for ax in axes[n:]: ax.axis('off')
    plt.suptitle(f'Average Confusion Matrices ({NUMBER_OF_FOLDS}-Fold)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_avg_confusion_matrices.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_normalised_confusion_matrices(cv_results, acts, class_names, cv_dir):
    n    = len(acts)
    cols = min(3, n); rows = (n + cols - 1) // cols
    nc   = len(class_names)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).ravel()
    tick_l = class_names if nc <= 10 else range(nc)
    for ax, act in zip(axes, acts):
        avg_cm = np.mean(cv_results[act]['fold_cms'], axis=0)
        norm   = avg_cm / (avg_cm.sum(axis=1, keepdims=True) + 1e-9)
        sns.heatmap(norm, annot=(nc <= 10), fmt='.2f', cmap='YlOrRd', ax=ax, cbar=True,
                    xticklabels=tick_l, yticklabels=tick_l, vmin=0, vmax=1, linewidths=0.3)
        ax.set_title(ACT_LABELS.get(act, act), fontweight='bold')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        ax.tick_params(axis='x', rotation=45, labelsize=6 if nc > 10 else 9)
        ax.tick_params(axis='y', rotation=0,  labelsize=6 if nc > 10 else 9)
    for ax in axes[n:]: ax.axis('off')
    plt.suptitle(f'Normalised Confusion Matrices ({NUMBER_OF_FOLDS}-Fold)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_normalised_confusion_matrices.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_roc_pr(cv_results, acts, num_classes, cv_dir):
    colours = _cv_colours(acts)
    fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(16, 6))

    for c, act in zip(colours, acts):
        y_true = np.concatenate([t for t, _ in cv_results[act]['fold_probs']])
        y_prob = np.concatenate([p for _, p in cv_results[act]['fold_probs']])
        y_bin  = label_binarize(y_true, classes=np.arange(num_classes))
        lb     = ACT_LABELS.get(act, act)

        if num_classes == 2:
            fpr_a, tpr_a, _ = roc_curve(y_bin.ravel(), y_prob[:, 1])
            auc_a = auc(fpr_a, tpr_a)
            p_a, r_a, _ = precision_recall_curve(y_bin.ravel(), y_prob[:, 1])
            ap_a = average_precision_score(y_bin.ravel(), y_prob[:, 1])
        else:
            fpr_a, tpr_a, _ = roc_curve(y_bin.ravel(), y_prob.ravel())
            auc_a = auc(fpr_a, tpr_a)
            p_a, r_a, _ = precision_recall_curve(y_bin.ravel(), y_prob.ravel())
            ap_a = average_precision_score(y_bin, y_prob, average='micro')

        ax_roc.plot(fpr_a, tpr_a, color=c, lw=2, label=f'{lb} (AUC={auc_a:.3f})')
        ax_pr.plot(r_a,   p_a,    color=c, lw=2, label=f'{lb} (AP={ap_a:.3f})')

    ax_roc.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
    ax_roc.set(xlim=[0, 1], ylim=[0, 1.05], xlabel='FPR', ylabel='TPR',
               title=f'Micro-avg ROC — Pooled {NUMBER_OF_FOLDS}-Fold')
    ax_roc.legend(fontsize=9); ax_roc.grid(True, alpha=0.3)

    ax_pr.set(xlim=[0, 1], ylim=[0, 1.05], xlabel='Recall', ylabel='Precision',
              title=f'Micro-avg PR — Pooled {NUMBER_OF_FOLDS}-Fold')
    ax_pr.legend(fontsize=9); ax_pr.grid(True, alpha=0.3)

    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_roc_pr.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_per_class_f1_heatmap(cv_results, acts, class_names, num_classes, cv_dir):
    nc       = min(len(class_names), 30)
    tick_l   = class_names[:nc]
    f1_matrix = []
    for act in acts:
        y_true = np.concatenate([t for t, _ in cv_results[act]['fold_probs']])
        y_prob = np.concatenate([p for _, p in cv_results[act]['fold_probs']])
        y_pred = np.argmax(y_prob, axis=1)
        pcf1   = f1_score(y_true, y_pred, average=None, zero_division=0,
                          labels=list(range(num_classes)))
        f1_matrix.append(pcf1[:nc])

    f1_matrix = np.array(f1_matrix)
    fig, ax = plt.subplots(figsize=(max(12, nc // 2), max(4, len(acts) * 0.9)))
    sns.heatmap(f1_matrix, annot=(nc <= 20), fmt='.2f', cmap='YlOrRd',
                xticklabels=tick_l, yticklabels=[ACT_LABELS.get(a, a) for a in acts],
                ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f'Per-Class F1 Score — Pooled {NUMBER_OF_FOLDS}-Fold CV',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Class'); ax.set_ylabel('Activation')
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_per_class_f1_heatmap.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_summary_table(summary_df, acts, cv_dir):
    cols = ['Activation', 'TrAcc', 'TrLoss', 'ValAcc±std', 'ValAUC±std',
            'TestAcc±std', 'TestF1±std', 's/epoch']
    rows = []
    for act in acts:
        r = summary_df.loc[act]
        rows.append([
            ACT_LABELS.get(act, act),
            f"{r['train_accuracy_mean']*100:.2f}%",
            f"{r['train_loss_mean']:.4f}",
            f"{r['val_accuracy_mean']*100:.2f}±{r['val_accuracy_std']*100:.2f}%",
            f"{r['val_auc_mean']:.4f}±{r['val_auc_std']:.4f}",
            f"{r['test_accuracy_mean']*100:.2f}±{r['test_accuracy_std']*100:.2f}%",
            f"{r['test_f1_mean']:.4f}±{r['test_f1_std']:.4f}",
            f"{r['train_time_per_epoch_s']:.2f}",
        ])
    fig, ax = plt.subplots(figsize=(16, 1 + 0.55 * len(rows)))
    ax.axis('off')
    tbl = ax.table(cellText=rows, colLabels=cols, loc='center', cellLoc='center')
    tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1, 1.9)
    for j in range(len(cols)):
        tbl[(0, j)].set_facecolor('#2C3E50')
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    best_idx = np.argmax([summary_df.loc[a, 'test_accuracy_mean'] for a in acts])
    for j in range(len(cols)):
        tbl[(best_idx + 1, j)].set_facecolor('#D5F5E3')
    for i in range(1, len(rows) + 1):
        for j in range(len(cols)):
            if i != best_idx + 1 and i % 2 == 0:
                tbl[(i, j)].set_facecolor('#F2F3F4')
    plt.title(f'CV Summary Table — {NUMBER_OF_FOLDS}-Fold (Green = Best Test Acc)',
              fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_summary_table.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_plot_failed_predictions(cv_results, acts, test_loader, num_classes,
                                input_channels, class_names, device,
                                cv_dir, dmean, dstd):
    use_amp = USE_AMP and device.type == 'cuda'
    for act in acts:
        if not cv_results[act]['fold_models']:
            continue
        best_fold_idx = int(np.argmax(cv_results[act]['fold_accuracies']))
        best_state    = cv_results[act]['fold_models'][best_fold_idx]

        model = build_model(MODEL_TYPE, num_classes, input_channels, act)
        # state_dict already has _orig_mod. stripped at save time — load directly
        model.load_state_dict(best_state['state_dict'])
        model = model.to(device); model.eval()

        fail_imgs, fail_true, fail_pred, fail_probs_list = [], [], [], []
        with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
            for imgs, lbls in test_loader:
                imgs = imgs.to(device, non_blocking=True)
                out  = model(imgs)
                prb  = torch.softmax(out, 1)
                _, p = torch.max(out, 1)
                mask = (p.cpu() != lbls)
                for i in range(len(lbls)):
                    if mask[i]:
                        fail_imgs.append(imgs[i].cpu())
                        fail_true.append(lbls[i].item())
                        fail_pred.append(p[i].cpu().item())
                        fail_probs_list.append(prb[i].cpu().numpy())
                    if len(fail_imgs) >= 24: break
                if len(fail_imgs) >= 24: break

        n_f = len(fail_imgs)
        lb  = ACT_LABELS.get(act, act)
        if n_f == 0:
            print(f"  {lb}: no test failures."); del model; gc.collect(); continue

        cols_f = 6; rows_f = (n_f + cols_f - 1) // cols_f
        fig, axes = plt.subplots(rows_f, cols_f, figsize=(cols_f * 3, rows_f * 3.5))
        axes = np.array(axes).ravel()
        for i in range(n_f):
            img  = denorm(fail_imgs[i], dmean, dstd)
            conf = fail_probs_list[i][fail_pred[i]] * 100
            axes[i].imshow(img)
            axes[i].set_title(
                f'T:{class_names[fail_true[i]]}\nP:{class_names[fail_pred[i]]}\n{conf:.1f}%',
                fontsize=6, color='red', fontweight='bold')
            axes[i].axis('off')
        for i in range(n_f, len(axes)): axes[i].axis('off')
        plt.suptitle(f'{lb} — CV Failed Predictions (Best Fold {best_fold_idx+1})',
                     fontsize=11, fontweight='bold', color='darkred')
        plt.tight_layout()
        p = os.path.join(cv_dir, f'cv_{act}_failed_predictions.png')
        plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
        print(f"  Saved: {p} ({n_f} failures)")

        ckpt = os.path.join(cv_dir, f'cv_best_{act}_fold{best_fold_idx+1}.pth')
        torch.save({'activation': act, 'fold': best_fold_idx + 1,
                    'val_accuracy': best_state['val_acc'],
                    'model_state_dict': best_state['state_dict'],
                    'num_classes': num_classes, 'model_type': MODEL_TYPE}, ckpt)
        print(f"  Checkpoint: {ckpt}")

        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()


def cv_plot_metric_box_violin(cv_results, acts, cv_dir):
    metrics = [
        ('fold_accuracies',      'Validation Accuracy', 'tab:green'),
        ('fold_test_accuracies', 'Test Accuracy',        'tab:orange'),
        ('fold_aucs',            'Validation AUC',       'tab:blue'),
        ('fold_f1s',             'Validation Macro F1',  'tab:purple'),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.ravel()
    labels = [ACT_LABELS.get(a, a) for a in acts]

    for ax, (key, title, col) in zip(axes, metrics):
        data = [cv_results[a][key] for a in acts]
        bp   = ax.boxplot(data, patch_artist=True, labels=labels,
                          medianprops=dict(color='black', lw=2))
        for patch in bp['boxes']:
            patch.set_facecolor(col); patch.set_alpha(0.5)
        for i, (d, lbl) in enumerate(zip(data, labels), 1):
            ax.scatter([i] * len(d), d, color=col, s=30, zorder=3, alpha=0.8)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_ylabel(title); ax.grid(True, alpha=0.3, axis='y')
        ax.tick_params(axis='x', rotation=25, labelsize=9)

    plt.suptitle(f'Per-Fold Metric Distribution ({NUMBER_OF_FOLDS}-Fold CV)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    p = os.path.join(cv_dir, 'cv_boxplots.png')
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"  Saved: {p}")


def cv_print_console_summary(summary_df, acts):
    print("\n" + "="*110)
    print("CV FINAL CONSOLIDATED REPORT")
    hdr = (f"{'Activation':<14} | {'TrAcc':>7} | {'TrLoss':>8} | "
           f"{'ValAcc':>7} | {'ValLoss':>8} | {'TestAcc':>8} | "
           f"{'TestAUC':>8} | {'TestF1':>7} | {'Time/fold':>10} | {'s/ep':>6}")
    print(hdr); print("-" * len(hdr))
    for act in acts:
        r = summary_df.loc[act]
        print(f"{ACT_LABELS.get(act, act):<14} | "
              f"{r['train_accuracy_mean']*100:6.2f}% | "
              f"{r['train_loss_mean']:8.4f} | "
              f"{r['val_accuracy_mean']*100:6.2f}% | "
              f"{r['val_loss_mean']:8.4f} | "
              f"{r['test_accuracy_mean']*100:7.2f}% | "
              f"{r['test_auc_mean']:8.4f} | "
              f"{r['test_f1_mean']:7.4f} | "
              f"{r['train_time_mean_s']:10.2f} | "
              f"{r['train_time_per_epoch_s']:6.3f}")
    print("="*110)


# ============================================================================
#  MAIN PIPELINE
# ============================================================================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")
    if device.type == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        torch.backends.cudnn.benchmark = True

    base_dir    = make_dir(BASE_OUTPUT_DIR)
    compare_dir = make_dir(base_dir, 'comparison')

    print("\nLoading dataset...")
    (train_ds, valid_ds, test_ds,
     num_classes, input_channels, image_size,
     class_names, dmean, dstd) = load_datasets(DATASET_TYPE)

    dl_kwargs = dict(batch_size=BATCH_SIZE, num_workers=2,
                     pin_memory=True, persistent_workers=True)
    train_loader = torch.utils.data.DataLoader(train_ds, shuffle=True,  **dl_kwargs)
    valid_loader = torch.utils.data.DataLoader(valid_ds, shuffle=False, **dl_kwargs)
    test_loader  = torch.utils.data.DataLoader(test_ds,  shuffle=False, **dl_kwargs)

    print(f"Dataset: {DATASET_TYPE.upper()} | Classes: {num_classes} | "
          f"Train: {len(train_ds)} | Val: {len(valid_ds)} | Test: {len(test_ds)}")
    print(f"Image size fed to model: {image_size}×{image_size}")

    plot_activation_shapes(ACTIVATION_FUNCTIONS, compare_dir)

    all_history   = {}
    all_metrics   = {}
    all_cm        = {}
    all_micro_roc = {}

    # ====================================================================
    #  Train + evaluate each activation
    # ====================================================================
    for act in ACTIVATION_FUNCTIONS:
        act_dir = make_dir(base_dir, act)

        model, history = train_one_activation(
            act, train_loader, valid_loader,
            num_classes, input_channels, device, act_dir)
        all_history[act] = history

        plot_training_curves(history, act, act_dir)

        for split, loader in [('train', train_loader),
                               ('val',   valid_loader),
                               ('test',  test_loader)]:
            lbl, pred, prob = evaluate_model(model, loader, device, num_classes)
            plot_confusion_matrix_single(lbl, pred, class_names, act, act_dir, split)
            plot_roc_pr(lbl, prob, num_classes, act, act_dir, split)
            if split == 'test':
                test_lbl, test_pred, test_prob = lbl, pred, prob

        plot_predictions(model, test_loader, device, class_names, act,
                         act_dir, dmean, dstd, n=12)
        plot_failed_predictions(model, test_loader, device, class_names, act,
                                 act_dir, dmean, dstd, max_fail=20)
        plot_feature_maps(model, test_loader, device, class_names, act,
                          act_dir, dmean, dstd, n_samples=2)

        lb_bin           = label_binarize(test_lbl, classes=range(num_classes))
        fpr_m, tpr_m, _  = roc_curve(lb_bin.ravel(), test_prob.ravel())
        rauc             = auc(fpr_m, tpr_m)
        p_m, r_m, _      = precision_recall_curve(lb_bin.ravel(), test_prob.ravel())
        micro_ap         = average_precision_score(lb_bin, test_prob, average='micro')
        macro_f1         = f1_score(test_lbl, test_pred, average='macro', zero_division=0)
        per_class_f1     = f1_score(test_lbl, test_pred, average=None,    zero_division=0)
        test_acc         = 100 * np.mean(test_lbl == test_pred)
        val_acc          = history['val_acc'][-1]

        target    = 0.80 * val_acc
        conv_ep   = next((i + 1 for i, a in enumerate(history['val_acc']) if a >= target),
                         len(history['val_acc']))
        conv_speed = 1.0 - (conv_ep / len(history['val_acc']))

        all_metrics[act] = {
            'test_acc':          test_acc,
            'val_acc':           val_acc,
            'macro_f1':          macro_f1,
            'micro_roc_auc':     rauc,
            'micro_ap':          micro_ap,
            'per_class_f1':      per_class_f1.tolist(),
            'convergence_speed': conv_speed,
        }
        all_cm[act]        = confusion_matrix(test_lbl, test_pred)
        all_micro_roc[act] = (fpr_m, tpr_m, rauc)

        # Sstrip _orig_mod. prefix added by torch.compile before saving checkpoint
        sd = strip_compile_prefix(model.state_dict())
        ckpt = os.path.join(act_dir, f'model_{act}.pth')
        torch.save({'state_dict': sd,
                    'activation': act, 'num_classes': num_classes,
                    'class_names': class_names}, ckpt)
        print(f"  Model checkpoint: {ckpt}")

        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ====================================================================
    #  Cross-activation comparison plots
    # ====================================================================
    print("\n" + "="*60)
    print("  Generating cross-activation comparison visualisations...")
    print("="*60)

    plot_comparison_training_curves(all_history, compare_dir)
    plot_learning_curve_all(all_history, compare_dir)
    plot_epoch_time_boxplot(all_history, compare_dir)
    plot_comparison_bar(all_metrics, 'test_acc',      'Test Accuracy Comparison',
                        'Accuracy (%)', compare_dir, 'comparison_test_accuracy.png')
    plot_comparison_bar(all_metrics, 'macro_f1',      'Macro F1 Comparison',
                        'F1 Score',    compare_dir, 'comparison_macro_f1.png')
    plot_comparison_bar(all_metrics, 'micro_roc_auc', 'Micro-avg ROC AUC Comparison',
                        'AUC',         compare_dir, 'comparison_roc_auc.png')
    plot_comparison_bar(all_metrics, 'micro_ap',      'Micro-avg Average Precision Comparison',
                        'AP',          compare_dir, 'comparison_avg_precision.png')
    plot_comparison_heatmap(all_metrics, class_names, compare_dir)
    plot_radar(all_metrics, compare_dir)
    plot_comparison_confusion_heatmap(all_cm, class_names, compare_dir)
    plot_comparison_roc_overlay(all_micro_roc, compare_dir)
    plot_final_metric_table(all_metrics, compare_dir)

    results_serializable = {}
    for act, m in all_metrics.items():
        results_serializable[act] = {k: (float(v) if isinstance(v, float) else v)
                                      for k, v in m.items()}
    with open(os.path.join(compare_dir, 'metrics_summary.json'), 'w') as f:
        json.dump(results_serializable, f, indent=2)

    print("\n" + "="*60)
    print("  FINAL RESULTS SUMMARY")
    print("="*60)
    header = (f"{'Activation':<14} {'Test Acc':>9} {'Val Acc':>9} "
              f"{'Macro F1':>10} {'ROC-AUC':>9} {'Avg Prec':>10}")
    print(header); print("-" * len(header))
    for act in ACTIVATION_FUNCTIONS:
        m  = all_metrics[act]
        lb = ACT_LABELS.get(act, act)
        print(f"{lb:<14} {m['test_acc']:>8.2f}% {m['val_acc']:>8.2f}%"
              f" {m['macro_f1']:>10.4f} {m['micro_roc_auc']:>9.4f} {m['micro_ap']:>10.4f}")

    best = max(all_metrics, key=lambda a: all_metrics[a]['test_acc'])
    print(f"\n  Best activation: {ACT_LABELS.get(best, best)} "
          f"(Test Acc = {all_metrics[best]['test_acc']:.2f}%)")
    print(f"\n  All results saved to: {base_dir}")

    # ====================================================================
    #  K-FOLD CROSS-VALIDATION COMPARISON
    # ====================================================================
    if RUN_CV_COMPARISON:
        cv_dir = make_dir(base_dir, 'cv_comparison')

        cv_results, summary_df = run_cv_comparison(
            train_ds, test_loader, num_classes, input_channels,
            class_names, device, cv_dir, dmean, dstd)

        acts = CV_ACTIVATION_FUNCTIONS
        print("\n" + "="*60)
        print("  Generating CV visualisations...")
        print("="*60)

        cv_plot_bar(summary_df, acts, 'val_accuracy_mean',   'val_accuracy_std',
                    f'Mean Val Accuracy (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'Accuracy', 'lightgreen',    'darkgreen',  cv_dir, 'cv_val_accuracy.png')
        cv_plot_bar(summary_df, acts, 'val_loss_mean',        'val_loss_std',
                    f'Mean Val Loss (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'Loss',     'salmon',        'darkred',    cv_dir, 'cv_val_loss.png')
        cv_plot_bar(summary_df, acts, 'train_accuracy_mean',  'train_accuracy_std',
                    f'Mean Train Accuracy (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'Accuracy', 'cornflowerblue','navy',       cv_dir, 'cv_train_accuracy.png')
        cv_plot_bar(summary_df, acts, 'test_accuracy_mean',   'test_accuracy_std',
                    f'Mean Test Accuracy (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'Accuracy', 'gold',          'darkorange', cv_dir, 'cv_test_accuracy.png')
        cv_plot_bar(summary_df, acts, 'test_loss_mean',       'test_loss_std',
                    f'Mean Test Loss (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'Loss',     'tomato',        'firebrick',  cv_dir, 'cv_test_loss.png')
        cv_plot_bar(summary_df, acts, 'val_auc_mean',         'val_auc_std',
                    f'Mean Val AUC (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'AUC',      'mediumpurple',  'indigo',     cv_dir, 'cv_val_auc.png')
        cv_plot_bar(summary_df, acts, 'val_f1_mean',          'val_f1_std',
                    f'Mean Val Macro F1 (±1 std) — {NUMBER_OF_FOLDS}-Fold',
                    'F1',       'lightcoral',    'crimson',    cv_dir, 'cv_val_f1.png')

        cv_plot_training_time(summary_df, acts, cv_dir)
        cv_plot_fold_heatmap(cv_results, acts, 'fold_accuracies',
                             f'Fold-wise Val Accuracy (%) — {NUMBER_OF_FOLDS}-Fold',
                             'YlGnBu', 'Val Accuracy (%)',
                             cv_dir, 'cv_heatmap_val_accuracy.png', scale=100.0)
        cv_plot_fold_heatmap(cv_results, acts, 'fold_test_accuracies',
                             f'Fold-wise Test Accuracy (%) — {NUMBER_OF_FOLDS}-Fold',
                             'OrRd', 'Test Accuracy (%)',
                             cv_dir, 'cv_heatmap_test_accuracy.png', scale=100.0)
        cv_plot_fold_heatmap(cv_results, acts, 'fold_aucs',
                             f'Fold-wise Val AUC — {NUMBER_OF_FOLDS}-Fold',
                             'PuBu', 'AUC',
                             cv_dir, 'cv_heatmap_val_auc.png', scale=1.0)
        cv_plot_fold_heatmap(cv_results, acts, 'fold_f1s',
                             f'Fold-wise Val F1 — {NUMBER_OF_FOLDS}-Fold',
                             'RdPu', 'Macro F1',
                             cv_dir, 'cv_heatmap_val_f1.png', scale=1.0)
        cv_plot_fold_heatmap(cv_results, acts, 'fold_train_times',
                             f'Fold-wise Training Time (s) — {NUMBER_OF_FOLDS}-Fold',
                             'Blues', 'Time (s)',
                             cv_dir, 'cv_heatmap_train_time.png', scale=1.0)

        cv_plot_learning_curves(cv_results, acts, cv_dir)
        cv_plot_individual_fold_curves(cv_results, acts, cv_dir)
        cv_plot_avg_confusion_matrices(cv_results, acts, class_names, cv_dir)
        cv_plot_normalised_confusion_matrices(cv_results, acts, class_names, cv_dir)
        cv_plot_roc_pr(cv_results, acts, num_classes, cv_dir)
        cv_plot_per_class_f1_heatmap(cv_results, acts, class_names, num_classes, cv_dir)
        cv_plot_radar(summary_df, acts, cv_dir)
        cv_plot_metric_box_violin(cv_results, acts, cv_dir)
        cv_plot_summary_table(summary_df, acts, cv_dir)
        cv_plot_failed_predictions(cv_results, acts, test_loader,
                                    num_classes, input_channels, class_names,
                                    device, cv_dir, dmean, dstd)

        cv_print_console_summary(summary_df, acts)
        print(f"\n  CV results saved to: {cv_dir}")


if __name__ == '__main__':
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Model: RESNET34
DATASET: TINYIMAGENET200
SINGLE MODEL NUM EPOCHS: 50
CV NUM EPOCHS: 40
BATCH SIZE: 128
SINGLE MODEL LEARNING RATE: 0.01
CV LEARNING RATE: 0.01
OPTIMIZER: SGD
EARLY STOPPING PATIENCE: 5
AMP (Mixed Precision): True

Device: cuda
GPU: NVIDIA L4

Loading dataset...
Dataset: TINYIMAGENET200 | Classes: 200 | Train: 100000 | Val: 10000 | Test: 10000
Image size fed to model: 32×32
  Saved: /content/drive/MyDrive/ACTIVATION_FXTNS_RESULTS/comparison/activation_shapes.png

  Training with: RALU
  Epoch   1/50  TrainL=4.6819  ValL=4.0849  TrainAcc=7.25%  ValAcc=13.74%  LR=0.009990  Time=153.9s
  Epoch   2/50  TrainL=3.8776  ValL=3.6374  TrainAcc=16.36%  ValAcc=19.99%  LR=0.009961  Time=39.2s
  Epoch   3/50  TrainL=3.5632  ValL=3.4735  TrainAcc=21.05%  ValAcc=23.10%  LR=0.009911  Time=39.1s
  Epoch   4/50  TrainL=3.3584  ValL=3.3086  TrainAcc=24.46%  ValA

W0414 12:30:15.735000 8063 torch/_dynamo/convert_frame.py:1676] [2/8] torch._dynamo hit config.recompile_limit (8)
W0414 12:30:15.735000 8063 torch/_dynamo/convert_frame.py:1676] [2/8]    function: 'forward' (/tmp/ipykernel_8063/2018633510.py:597)
W0414 12:30:15.735000 8063 torch/_dynamo/convert_frame.py:1676] [2/8]    last reason: 2/7: GLOBAL_STATE changed: grad_mode 
W0414 12:30:15.735000 8063 torch/_dynamo/convert_frame.py:1676] [2/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0414 12:30:15.735000 8063 torch/_dynamo/convert_frame.py:1676] [2/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/compile/programming_model.recompilation.html


  Epoch   1/50  TrainL=4.7327  ValL=4.1797  TrainAcc=6.45%  ValAcc=12.88%  LR=0.009990  Time=79.1s
  Epoch   2/50  TrainL=3.9395  ValL=3.6939  TrainAcc=15.55%  ValAcc=19.08%  LR=0.009961  Time=39.3s
  Epoch   3/50  TrainL=3.6191  ValL=3.4701  TrainAcc=20.33%  ValAcc=23.04%  LR=0.009911  Time=39.0s
  Epoch   4/50  TrainL=3.4204  ValL=3.3850  TrainAcc=23.57%  ValAcc=24.13%  LR=0.009843  Time=39.3s
  Epoch   5/50  TrainL=3.2687  ValL=3.2454  TrainAcc=26.15%  ValAcc=26.14%  LR=0.009755  Time=40.3s
  Epoch   6/50  TrainL=3.1491  ValL=3.1619  TrainAcc=28.13%  ValAcc=28.00%  LR=0.009649  Time=39.5s
  Epoch   7/50  TrainL=3.0512  ValL=3.0654  TrainAcc=30.00%  ValAcc=30.54%  LR=0.009524  Time=39.2s
  Epoch   8/50  TrainL=2.9627  ValL=3.0043  TrainAcc=31.57%  ValAcc=30.96%  LR=0.009382  Time=39.1s
  Epoch   9/50  TrainL=2.8908  ValL=2.9847  TrainAcc=33.01%  ValAcc=31.52%  LR=0.009222  Time=38.8s
  Epoch  10/50  TrainL=2.8270  ValL=2.8882  TrainAcc=34.19%  ValAcc=33.09%  LR=0.009045  Time=39.1s
 

# End